# Conditional-NF Novel View Generator -- Colab training

Trains a **conditional Normalizing Flow** that models `P((3D point, 3D direction) | DINO feature)`,
against a **frozen** nerfacto NeRF (trained once here, then never fine-tuned again).

This repo is a public fork (`itayhanoch/VF-NeRF-conditional`), so no GitHub token is needed to clone it.

Steps: install deps (including compiling tiny-cuda-nn from source) -> mount Drive -> download/prepare
a scene -> train the frozen NeRF backbone -> precompute DINO features -> train the conditional NF,
checkpointing to Drive so it survives disconnects. `Runtime` -> `Run all` once `TRAIN_RESUME_MODE` (below)
is set.


In [ ]:
#@title Install dependencies (CUDA 11.7 toolchain + Python 3.10 venv + tiny-cuda-nn)
import os

REPO_URL = "https://github.com/itayhanoch/VF-NeRF-conditional.git"

# Colab's system Python is newer than the wheels available for torch==1.13.1
# (needed for tiny-cuda-nn/nerfacc compatibility), so build an isolated
# Python 3.10 venv instead of installing into the system interpreter.
!apt-get -qq install -y python3.10-venv cuda-nvcc-11-7 cuda-nvrtc-dev-11-7 \
    libcublas-dev-11-7 libcufft-dev-11-7 libcurand-dev-11-7 \
    libcusolver-dev-11-7 libcusparse-dev-11-7 libnpp-dev-11-7 libnvjpeg-dev-11-7 \
    ninja-build

# So the compiled tiny-cuda-nn extension can find libnvrtc.so.11.2 at runtime.
!echo '/usr/local/cuda-11.7/lib64' > /etc/ld.so.conf.d/cuda-11-7.conf && ldconfig

if not os.path.isdir("/content/venv310"):
    !python3.10 -m venv /content/venv310
VENV = "/content/venv310/bin"

# setuptools>=81 dropped pkg_resources (breaks several deps); wheel is
# required for the --no-build-isolation installs below.
!{VENV}/pip install --quiet 'setuptools<81' wheel
!{VENV}/pip install --quiet torch==1.13.1 torchvision functorch --extra-index-url https://download.pytorch.org/whl/cu117
!{VENV}/pip install --quiet ninja

os.environ["CUDA_HOME"] = "/usr/local/cuda-11.7"
os.environ["PATH"] = "/usr/local/cuda-11.7/bin:" + os.environ["PATH"]
# --no-build-isolation: an isolated build env can't see the torch just installed above.
!{VENV}/pip install --quiet --no-build-isolation "git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch"

if not os.path.isdir("VF-NeRF-conditional"):
    !git clone --quiet {REPO_URL}
%cd VF-NeRF-conditional

# Known upstream bug: one eval call site is missing a required `step` arg,
# which crashes training at the first periodic full-image eval (~step 25000).
!sed -i 's/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch)/metrics_dict, _ = self.model.get_image_metrics_and_images(outputs, batch, step)/' \
    nerfstudio/pipelines/base_pipeline.py

!{VENV}/pip install --quiet --no-build-isolation -e . -e ./normalizing-flows
!{VENV}/pip install --quiet remotezip


In [ ]:
#@title Mount Google Drive (checkpoints persist here across disconnects)
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/vf_nerf_conditional"
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)


## Configuration

Set `TRAIN_RESUME_MODE` once, then `Run all` -- nothing later needs input.
Default target scene is `bonsai` from the Mip-NeRF 360 dataset (a real,
casually-captured, 360 degree, object-centric scene -- a stand-in for VF-NeRF's
own unreleased "table" scene; see the README for why). Dataset-agnostic: point
`SCENE_NAME`/`DATA_DIR` at your own COLMAP-processed scene later with no code
changes.


In [ ]:
#@title Configuration
SCENE_NAME = "bonsai"  #@param ["bonsai", "counter", "kitchen", "room"]
DATA_DIR = f"data/mipnerf360/{SCENE_NAME}"
NERF_OUTPUT_DIR = f"{DRIVE_ROOT}/nerf_outputs"
COND_NF_CHECKPOINT_DIR = f"{DRIVE_ROOT}/conditional_nf/{SCENE_NAME}"

# "finetune": resume+continue training if a checkpoint already exists on Drive
# "reset": always retrain from scratch
# "skip": load existing weights only, skip training entirely
TRAIN_RESUME_MODE = "finetune"  #@param ["finetune", "reset", "skip"]


In [ ]:
#@title Download the example scene (skip if DATA_DIR already exists, e.g. your own scene)
import os

if not os.path.isdir(DATA_DIR):
    # Only fetches this one scene's files via HTTP range requests, not the full
    # ~12.5GB archive bundling all 9 Mip-NeRF 360 scenes -- see README for why
    # this is used instead of nerfstudio's own (currently unreliable) example
    # captures.
    !{VENV}/python scripts/downloads/download_mipnerf360.py --scene {SCENE_NAME} --save-dir data/mipnerf360
else:
    print(f"{DATA_DIR} already exists, skipping download.")


In [ ]:
#@title Generate downscaled training images (avoids Colab OOM on full-res caching)
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

DOWNSCALE_FACTOR = 4

src_dir = Path(DATA_DIR) / "images"
dst_dir = Path(DATA_DIR) / f"images_{DOWNSCALE_FACTOR}"

if not dst_dir.is_dir():
    dst_dir.mkdir(exist_ok=True)
    files = sorted(src_dir.glob("*"))
    print(f"Downscaling {len(files)} images -> {dst_dir}")

    def _process(f):
        img = Image.open(f)
        w, h = img.size
        img = img.resize((w // DOWNSCALE_FACTOR, h // DOWNSCALE_FACTOR), Image.LANCZOS)
        img.save(dst_dir / f.name)

    with ThreadPoolExecutor(max_workers=8) as ex:
        list(ex.map(_process, files))
    print(f"Done: {len(list(dst_dir.glob('*')))} files")
else:
    print(f"{dst_dir} already exists, skipping.")


#@title Train (or reuse) the frozen nerfacto backbone
import glob
import os

# So the venv's tiny-cuda-nn extension can find libnvrtc.so.11.2 at runtime.
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-11.7/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

existing = sorted(glob.glob(f"{NERF_OUTPUT_DIR}/{SCENE_NAME}/nerfacto/*/config.yml"))

if existing and TRAIN_RESUME_MODE == "skip":
    NERF_CONFIG = existing[-1]
    print(f"Using existing frozen-NeRF checkpoint: {NERF_CONFIG}")
else:
    # --downscale-factor must match the images_{N} folder generated above --
    # training on full-res images OOMs on Colab's ~12GB memory limit.
    !{VENV}/ns-train nerfacto \
        --data {DATA_DIR} \
        --output-dir {NERF_OUTPUT_DIR} \
        --viewer.quit-on-train-completion True \
        --vis tensorboard \
        nerfstudio-data --downscale-factor {DOWNSCALE_FACTOR}
    existing = sorted(glob.glob(f"{NERF_OUTPUT_DIR}/{SCENE_NAME}/nerfacto/*/config.yml"))
    NERF_CONFIG = existing[-1]
    print(f"Trained frozen-NeRF checkpoint: {NERF_CONFIG}")


In [ ]:
#@title Train (or reuse) the frozen nerfacto backbone
import glob

existing = sorted(glob.glob(f"{NERF_OUTPUT_DIR}/{SCENE_NAME}/nerfacto/*/config.yml"))

if existing and TRAIN_RESUME_MODE == "skip":
    NERF_CONFIG = existing[-1]
    print(f"Using existing frozen-NeRF checkpoint: {NERF_CONFIG}")
else:
    !ns-train nerfacto \
        --data {DATA_DIR} \
        --output-dir {NERF_OUTPUT_DIR} \
        --viewer.quit-on-train-completion True \
        --vis tensorboard
    existing = sorted(glob.glob(f"{NERF_OUTPUT_DIR}/{SCENE_NAME}/nerfacto/*/config.yml"))
    NERF_CONFIG = existing[-1]
    print(f"Trained frozen-NeRF checkpoint: {NERF_CONFIG}")


#@title Train the conditional NF
if TRAIN_RESUME_MODE == "skip":
    print("TRAIN_RESUME_MODE == 'skip': not training, expecting an existing checkpoint at "
          f"{COND_NF_CHECKPOINT_DIR}/latest.pt")
else:
    if TRAIN_RESUME_MODE == "reset":
        import shutil
        shutil.rmtree(COND_NF_CHECKPOINT_DIR, ignore_errors=True)

    !{VENV}/python scripts/train_conditional_nf.py \
        --nerf-config {NERF_CONFIG} \
        --scene-dir {DATA_DIR} \
        --checkpoint-dir {COND_NF_CHECKPOINT_DIR} \
        --max-steps 20000 \
        --batch-size 4096


In [ ]:
#@title Train the conditional NF
if TRAIN_RESUME_MODE == "skip":
    print("TRAIN_RESUME_MODE == 'skip': not training, expecting an existing checkpoint at "
          f"{COND_NF_CHECKPOINT_DIR}/latest.pt")
else:
    if TRAIN_RESUME_MODE == "reset":
        import shutil
        shutil.rmtree(COND_NF_CHECKPOINT_DIR, ignore_errors=True)

    !python scripts/train_conditional_nf.py \
        --nerf-config {NERF_CONFIG} \
        --scene-dir {DATA_DIR} \
        --checkpoint-dir {COND_NF_CHECKPOINT_DIR} \
        --max-steps 20000 \
        --batch-size 4096


#@title Quick sanity check
# Runs as a subprocess under the venv's interpreter -- this notebook's own
# kernel is Colab's system Python, which doesn't have nerfstudio/torch
# installed (everything above went into /content/venv310 instead).
sanity_script = f'''
import random
from pathlib import Path

import torch

from nerfstudio.data.dataparsers.nerfstudio_dataparser import NerfstudioDataParserConfig
from nerfstudio.fields.nf_field import ConditionalNFField
from nerfstudio.utils.dino_features import DinoExtractor, get_or_compute_cache, load_image_chw_01
from nerfstudio.utils.eval_utils import eval_setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_, pipeline, _, _ = eval_setup(Path("{NERF_CONFIG}"), test_mode="inference")
nerf_model = pipeline.model.to(device).eval()

dataparser = NerfstudioDataParserConfig(data=Path("{DATA_DIR}")).setup()
outputs = dataparser.get_dataparser_outputs(split="train")
cameras = outputs.cameras.to(device)

ckpt = torch.load("{COND_NF_CHECKPOINT_DIR}/latest.pt", map_location=device)
field = ConditionalNFField(
    context_dim=ckpt["context_dim"], num_dims=ckpt["num_dims"], num_blocks=ckpt["num_blocks"],
    hidden_dim=ckpt["hidden_dim"], cond_prior=ckpt["cond_prior"], use_cond_in_coupling=True,
    use_batchnorm=ckpt["use_batchnorm"], reduce_dim=ckpt.get("reduce_dim"),
    reduce_divide_factor=ckpt.get("reduce_divide_factor", 8), device=str(device),
)
field.load_state_dict(ckpt["model_state"])
field.eval()

extractor = DinoExtractor(model_name=ckpt["dino_model_name"], device=str(device))
image_path = random.choice(outputs.image_filenames)
img = load_image_chw_01(image_path)
feat_map = get_or_compute_cache(img, image_path.stem, Path("{DATA_DIR}") / "dino_cache", extractor)
_, h, w = img.shape
y, x = h // 2, w // 2
condition = feat_map[:, y, x]

with torch.no_grad():
    samples = field.sample(num_samples=100, context=condition)
    log_p = field.log_prob(samples, condition.unsqueeze(0).expand(100, -1)).squeeze(-1)
best = samples[log_p.argmax()]
print(f"Best sample (position, direction): {{best.tolist()}}")
print(f"log-likelihood: {{log_p.max().item():.3f}}")
'''

with open("/content/sanity_check.py", "w") as fh:
    fh.write(sanity_script)

!{VENV}/python /content/sanity_check.py


In [ ]:
#@title Quick sanity check
import random
from pathlib import Path

import torch

from nerfstudio.data.dataparsers.nerfstudio_dataparser import NerfstudioDataParserConfig
from nerfstudio.fields.nf_field import ConditionalNFField
from nerfstudio.utils.dino_features import DinoExtractor, get_or_compute_cache, load_image_chw_01
from nerfstudio.utils.eval_utils import eval_setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_, pipeline, _, _ = eval_setup(Path(NERF_CONFIG), test_mode="inference")
nerf_model = pipeline.model.to(device).eval()

dataparser = NerfstudioDataParserConfig(data=Path(DATA_DIR)).setup()
outputs = dataparser.get_dataparser_outputs(split="train")
cameras = outputs.cameras.to(device)

ckpt = torch.load(f"{COND_NF_CHECKPOINT_DIR}/latest.pt", map_location=device)
field = ConditionalNFField(
    context_dim=ckpt["context_dim"], num_dims=ckpt["num_dims"], num_blocks=ckpt["num_blocks"],
    hidden_dim=ckpt["hidden_dim"], cond_prior=ckpt["cond_prior"], use_cond_in_coupling=True,
    use_batchnorm=ckpt["use_batchnorm"], reduce_dim=ckpt.get("reduce_dim"),
    reduce_divide_factor=ckpt.get("reduce_divide_factor", 8), device=str(device),
)
field.load_state_dict(ckpt["model_state"])
field.eval()

extractor = DinoExtractor(model_name=ckpt["dino_model_name"], device=str(device))
image_path = random.choice(outputs.image_filenames)
img = load_image_chw_01(image_path)
feat_map = get_or_compute_cache(img, image_path.stem, Path(DATA_DIR) / "dino_cache", extractor)
_, h, w = img.shape
y, x = h // 2, w // 2
condition = feat_map[:, y, x]

with torch.no_grad():
    samples = field.sample(num_samples=100, context=condition)
    log_p = field.log_prob(samples, condition.unsqueeze(0).expand(100, -1)).squeeze(-1)
best = samples[log_p.argmax()]
print(f"Best sample (position, direction): {best.tolist()}")
print(f"log-likelihood: {log_p.max().item():.3f}")
